# BetaTour: güven uyarlamalı aday listesi

Bu defter, eğitilmiş ağı yeniden eğitmeden LKH'ye verdiği aday listesini iyileştirmeyi sınar. Sabit yöntem her düğümde üç model adayı alır. Yeni yöntem modelin düğüm güvenine göre alt çeyrekte 2, orta yarıda 3, üst çeyrekte 4 model adayı alır ve her listeyi koşunun POPMUSIC adaylarıyla tam beşe doldurur. Kural sonuçlar görülmeden sabitlenmiştir; başarı garantisi değildir.

Birincil karşılaştırma güven uyarlamalı BetaTour ile sabit üç adaylı BetaTour arasındadır. İki kol aynı ağ ağırlıklarını, ilk turları, altı LKH tohumunu, aday sayısını ve süre sınırını kullanır. 40 yeni sentetik örneğin üretim tohumları makaledeki 51 örnekten ve önceki 40 örnekli defterden farklıdır.

Colab'da L4 veya A100 GPU, High RAM ve en az 12 vCPU seçip Tümünü çalıştır'ı başlatın. Defter MyDrive/BetaTour/adaptive40_v1 klasörünü oluşturur. Oturum kesilirse aynı GPU/vCPU türüyle yeniden Tümünü çalıştır'ı seçin. Tamamlanan örnekler atlanır; yeni sanal makinede yarım örnek baştan çalışır. Altı kollu koşu yaklaşık 7–10 saat sürebilir. Defterin son hücresi tamamlandığında ZIP Drive'a yazılır ve tarayıcı indirmesi başlar.


## 1. Sabit ayarlar ve koşu öncesi analiz planı

40 yeni sentetik örnek: uniform ve kümeli dağılımların her birinde 1.000 düğümden sekiz, 10.000 düğümden sekiz, 20.000 düğümden dört örnek. Üretim tohumları ve bütçeler manifestte kilitlenir.

Birincil sonlanım, her örnekte bütçe dolana kadar altı LKH tohumunun bulduğu en düşük tur maliyetidir. Birincil karşılaştırma uyarlamalı / sabit BetaTour'dur. Eşli yüzde fark 100 × (uyarlamalı − sabit) / sabit olarak hesaplanır; negatif değer uyarlamalı yöntem lehinedir. Tam 40 örnekte eşitlikleri dışlayan iki yönlü kesin işaret testi uygulanır (p < 0,05). Ortalama, medyan, W/T/L ve 5.000 örnek düzeyi bootstrap tekrarından %95 aralık da raporlanır. Altı LKH tohumu bağımsız örnek sayılmaz.

Uyarlamalı yöntemin eşit-beş POPMUSIC yeniden başlatma, tur sıklığı ve kesintisiz POPMUSIC ile karşılaştırmaları ikincildir ve keşifseldir. Sekiz genel karşılaştırmaya Holm düzeltmesi uygulanır. Alt gruplar yalnız betimleyicidir. Sonuçlardan sonra model, çeyrek oranı, 2/3/4 kuralı, bütçeler, tohumlar veya birincil sonlanım değiştirilmeyecektir. Eksik örnekli rapor ara sonuçtur.

MAX_NEW_INSTANCES bir oturumda işlenecek yeni örnek sayısını sınırlar.


In [ ]:
# Colab runtime: L4 or A100 GPU + High RAM, at least 12 vCPUs.
EXPERIMENT_NAME = "adaptive40_v1"
PROFILE = "adaptive40"
GROUPS = ["uni1k", "clu1k", "uni10k", "clu10k", "uni20k", "clu20k"]
INSTANCE_SPECS = [
    ["uni1k", 1000, 8, "uniform", 2101, 5.0],
    ["clu1k", 1000, 8, "clustered", 2202, 5.0],
    ["uni10k", 10000, 8, "uniform", 2303, 120.0],
    ["clu10k", 10000, 8, "clustered", 2404, 120.0],
    ["uni20k", 20000, 4, "uniform", 2505, 300.0],
    ["clu20k", 20000, 4, "clustered", 2606, 300.0],
]
MAX_NEW_INSTANCES = None
SAVE_TO_DRIVE = True
INCLUDE_CANDIDATES_IN_ZIP = False

W = 6
REQUIRE_12_VCPUS = True
SEEDS = list(range(101, 107))
K_GRAPH = 16
K_LIST = 5
K_LEARNED = 3
ADAPTIVE_FRACTION = 0.25
ADAPTIVE_K = [2, 3, 4]
PAD_SECONDS = 3.0
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"

assert W == len(SEEDS) and K_LEARNED == 3 and K_LIST == 5
assert ADAPTIVE_K == [2, 3, 4] and 0 < ADAPTIVE_FRACTION < 0.5
assert PROFILE == "adaptive40" and sum(spec[2] for spec in INSTANCE_SPECS) == 40

ANALYSIS_PLAN = {
    "unit": "instance",
    "sample_size": 40,
    "endpoint": "best tour cost among six fixed seeds at the time budget",
    "primary": "betatour_adaptive vs betatour",
    "effect": "mean paired 100*(betatour_adaptive-betatour)/betatour",
    "test": "two-sided exact sign test, ties excluded",
    "alpha": 0.05,
    "secondary": ["betatour_adaptive vs restart_pop_5",
                  "betatour_adaptive vs frequency", "betatour_adaptive vs pop_full"],
    "subgroups": "descriptive only",
    "post_result_tuning": "none",
    "confidence_ties": "middle 3 candidates",
}


## 2. Ortam, model, Drive ve LKH

Sabit Git commit'i ve model dosyasının SHA256 değeri doğrulanır. LKH 3.0.14 önce yazarın kaynağından, erişilemiyorsa SHA256 denetimli kaynak yedeğinden derlenir. Sonuç klasörü bu hücrede oluşturulur.


In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tarfile
import time
import urllib.request
import uuid
from pathlib import Path

for package, module in [("scipy", "scipy"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import scipy
import torch

torch.set_num_threads(1)
assert torch.cuda.is_available(), "Colab'da GPU runtime seçin: L4 veya A100."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"Altı LKH süreci için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Makaledeki 12 vCPU düzeni için 12 vCPU gerekli; görünen: {CPU_COUNT}"
if CPU_COUNT < 2 * W:
    print(f"UYARI: {CPU_COUNT} vCPU var. Bu çalışma makaledeki 12 vCPU düzenini tekrarlamıyor.")

def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

CPU_MODEL = cpu_model_name()

WORK_ROOT = Path("/content/betatour_validation_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
actual_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
assert actual_hash == CHECKPOINT_SHA256, "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)  # pinned, trusted repository
assert MODEL.cfg.get("use_ensemble") is True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BETATOUR = Path("/content/drive/MyDrive/BetaTour")
    DRIVE_BETATOUR.mkdir(parents=True, exist_ok=True)
    OUTPUT_ROOT = DRIVE_BETATOUR / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "stages", "candidates", "errors", "summary", "references"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": T.env_info().get("ram_gb"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
}
MANIFEST = {
    "schema": 1,
    "experiment_name": EXPERIMENT_NAME,
    "profile": PROFILE,
    "groups": GROUPS,
    "instance_specs": INSTANCE_SPECS,
    "adaptive_rule": {"fraction": ADAPTIVE_FRACTION, "k": ADAPTIVE_K,
                      "tie_policy": "middle_3"},
    "seeds": SEEDS,
    "workers": W,
    "graph_k": K_GRAPH,
    "candidate_k": K_LIST,
    "learned_k": K_LEARNED,
    "repo_commit": REPO_COMMIT,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "hardware": HARDWARE,
    "arms": ["pop_full", "restart_pop_raw", "restart_pop_5", "frequency", "betatour", "betatour_adaptive"],
    "analysis_plan": ANALYSIS_PLAN,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "profile", "groups", "instance_specs", "adaptive_rule", "seeds", "workers", "graph_k",
                "candidate_k", "learned_k", "repo_commit", "checkpoint_sha256", "arms", "analysis_plan"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı EXPERIMENT_NAME kullanın."
    for key in ["gpu", "vcpus_visible"]:
        assert saved["hardware"][key] == HARDWARE[key], \
            f"Donanım {key} değişti. Eşli süre karşılaştırması için aynı Colab türünü seçin."
    if saved["hardware"] != HARDWARE:
        print("UYARI: CPU modeli, RAM veya kütüphane sürümleri değişmiş; her örneğin donanımı ayrıca kaydedilecek.")
else:
    manifest_path.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

# This local marker survives a notebook rerun on the same Colab VM. A fresh VM
# gets a new marker: incomplete instances are then rerun on one machine.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

def get_lkh_3014():
    """Official source first; pinned SHA256 checked source mirror if it is unavailable."""
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    if binary.exists():
        return str(binary)
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("resmi", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("SHA256 denetimli yedek", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH yedek SHA256 uyuşmazlığı: {got}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            build = subprocess.run(["make", "-j", str(min(4, CPU_COUNT))],
                                   cwd=root / "LKH-3.0.14", stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, timeout=600)
            if build.returncode != 0:
                raise RuntimeError("LKH derlemesi başarısız:\n" + build.stdout[-3000:])
            assert binary.exists(), "LKH ikilisi oluşmadı"
            return str(binary)
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi/derlenemedi: {last_error}")

LKH = get_lkh_3014()
print("Hazır:", GPU_NAME, "|", CPU_COUNT, "vCPU | model", sum(p.numel() for p in MODEL.parameters()),
      "parametre | LKH", LKH, "| sonuçlar", OUTPUT_ROOT)


Mounted at /content/drive
LKH kaynağı: resmi https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz
resmi başarısız: <urlopen error timed out>
LKH kaynağı: SHA256 denetimli yedek https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar
Hazır: NVIDIA A100-SXM4-80GB | 12 vCPU | model 1875787 parametre | LKH /content/betatour_validation_work/lkh/LKH-3.0.14/LKH | sonuçlar /content/drive/MyDrive/BetaTour/adaptive40_v1


## 3. Yeni örnekler

adaptive40_v1 örneklerinin üretim tohumları 2101–2606'dır. Önceki makale ve holdout örneklerinden farklıdır. TSPLIB ve evrilmiş zor örnekler bu sette yoktur; dış geçerlilik için ayrıca sınanmalıdır.


In [ ]:
def make_instances():
    specs = INSTANCE_SPECS
    result = []
    for group, n, count, dist, gen_seed, budget in specs:
        rng = np.random.default_rng(gen_seed)
        for i in range(count):
            xy = T.to_int_coords(T.gen_instance(n, dist, rng))
            result.append(dict(group=group, name=f"adaptive_{group}_{i:03d}", xy=xy,
                               ew="EUC_2D", problem_file=None, budget=budget, optimum=None))
    result.sort(key=lambda x: (x["budget"], x["name"]))
    assert len(result) == 40 and len({x["name"] for x in result}) == 40
    print(f"{len(result)} örnek hazır; tek kol bütçe toplamı {sum(x['budget'] for x in result)/3600:.2f} saat")
    return result

INSTANCES = make_instances()


40 örnek hazır; tek kol bütçe toplamı 1.22 saat


## 4. Güven uyarlamalı aday kuralı ve eşli deney

Altı kol: kesintisiz POPMUSIC; ham ve eşit beş adaylı POPMUSIC yeniden başlatmaları; tur sıklığı + POPMUSIC; sabit üç model adayı + POPMUSIC; güven uyarlamalı BetaTour. Beş yeniden başlatma kolu aynı altı ilk turu paylaşır.

Uyarlamalı yöntemde model güveninin %25 ve %75 eşikleri hesaplanır. Alt eşiğin altındakilerin ilk iki, üst eşiğin üstündekilerin ilk dört, aradakilerin ilk üç model adayı alınır. Eşiğe eşit puanlar ortadaki üç adaylı grupta kalır. Her tohumun POPMUSIC listesiyle beşe tamamlanır; eksik kalırsa model sırası kullanılır. Her kolun liste hazırlama ve devam süresi bütçesinden düşülür. Drive kayıt süresi algoritma çizelgesine eklenmez.

Ham POPMUSIC / kesintisiz POPMUSIC farkı saf yeniden başlatma etkisi değildir: aday dosyası ve cezalar da değişir.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import shutil
import traceback

POP_PARAMS = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST}
ARM_NAMES = ["pop_full", "restart_pop_raw", "restart_pop_5", "frequency", "betatour", "betatour_adaptive"]

def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)

def parallel_tasks(func, kwargs):
    with ThreadPoolExecutor(max_workers=W) as pool:
        futures = [pool.submit(func, **kw) for kw in kwargs]
        return [future.result() for future in futures]

def instance_common(inst):
    return dict(lkh_bin=LKH, workroot=str(WORK_ROOT / "lkh_tmp"), xy=inst["xy"],
                problem_file=inst["problem_file"], ew=inst["ew"])

def event_list(res, offset):
    return [[int(cost), float(t)] for cost, t in T.abs_events(res, offset=offset)]

def best_within(events, budget):
    costs = [int(cost) for cost, t in events if t <= budget + 1e-9]
    return min(costs) if costs else None

def make_full_arm(inst):
    budget = inst["budget"]
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_anytime, [dict(common, budget=budget, seed=seed,
                                             params=POP_PARAMS, timeout=budget + PAD_SECONDS)
                                          for seed in SEEDS])
    runs = []
    for seed, res in zip(SEEDS, raw):
        assert res["returncode"] in (0, None) and (res["returncode"] == 0 or res["timed_out"]), (seed, res)
        events = event_list(res, res["t_start"] - start)
        if not events and res["cost_min"] is not None and res["wall"] <= budget:
            events = [[int(res["cost_min"]), float(res["t_start"] - start + res["wall"])]]
        best = best_within(events, budget)
        assert best is not None, f"Kesintisiz POPMUSIC: tohum {seed} için bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, best_at_budget=int(best),
                         wall_s=float(res["wall"]), returncode=res["returncode"],
                         timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                wall_s=float(time.time() - start))

def make_phase1(inst):
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_first_trial, [dict(common, seed=seed,
                                                  timeout=inst["budget"] + PAD_SECONDS)
                                               for seed in SEEDS])
    phase_wall = time.time() - start
    n = len(inst["xy"])
    tours, cands, costs, first_events = [], [], [], []
    for seed, res in zip(SEEDS, raw):
        assert T.is_valid_tour(res["tour"], n), f"İlk tur geçersiz: {seed}"
        assert res["cand"].shape == (n, K_LIST), f"Aday boyutu yanlış: {seed}"
        events = event_list(res, res["t_start"] - start)
        if not events:
            events = [[int(res["cost"]), float(res["t_start"] - start + res["wall"])]]
        assert best_within(events, inst["budget"]) is not None, f"İlk tur bütçe dışında: {seed}"
        tours.append(res["tour"])
        cands.append(res["cand"])
        costs.append(res["cost"])
        first_events.append(events)
    arrays = dict(tours=np.stack(tours).astype(np.int32),
                  pop_candidates=np.stack(cands).astype(np.int32),
                  costs=np.asarray(costs, dtype=np.float64))
    meta = dict(wall_s=float(phase_wall), first_events=first_events,
                first_costs=[float(c) for c in costs])
    return arrays, meta

def valid_candidates(cands, n):
    assert cands.shape == (W, n, K_LIST), cands.shape
    assert np.all((cands >= 0) & (cands < n)), "Eksik/geçersiz aday bulundu"
    assert not np.any(cands == np.arange(n)[None, :, None]), "Düğüm kendi adayı olamaz"
    for i in range(K_LIST):
        for j in range(i + 1, K_LIST):
            assert not np.any(cands[:, :, i] == cands[:, :, j]), "Tekrarlı aday bulundu"

def make_candidate_lists(inst, phase):
    xy = inst["xy"]
    n = len(xy)
    tours, raw_pop, costs = phase["tours"], phase["pop_candidates"], phase["costs"]
    assert raw_pop.shape == (W, n, K_LIST)
    assert np.all((raw_pop == -1) | ((raw_pop >= 0) & (raw_pop < n)))
    assert not np.any(raw_pop == np.arange(n)[None, :, None])
    start = time.perf_counter()
    succ = np.stack([T.tour_succ_pred(tour)[0] for tour in tours], axis=1).astype(np.int32)
    successor_s = time.perf_counter() - start

    start = time.perf_counter()
    nearest, _ = T.knn(xy, K_GRAPH)
    pop5 = np.stack([T.mix_candidates(raw_pop[m], nearest, k_first=K_LIST,
                                      k=K_LIST, fill=nearest) for m in range(W)]).astype(np.int32)
    pop5_s = time.perf_counter() - start

    start = time.perf_counter()
    prediction = M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=costs)
    prediction_s = time.perf_counter() - start

    start = time.perf_counter()
    betatour = np.stack([T.mix_candidates(prediction["rank"], raw_pop[m], k_first=K_LEARNED,
                                          k=K_LIST, fill=prediction["rank"][:, K_LEARNED:])
                         for m in range(W)]).astype(np.int32)
    betatour_list_s = time.perf_counter() - start

    start = time.perf_counter()
    conf = np.asarray(prediction["conf"], dtype=np.float64)
    rank = np.asarray(prediction["rank"], dtype=np.int32)
    assert conf.shape == (n,) and np.isfinite(conf).all(), "Model güveni geçersiz"
    assert rank.shape == (n, K_GRAPH), "Model sıra boyutu yanlış"
    low = float(np.quantile(conf, ADAPTIVE_FRACTION))
    high = float(np.quantile(conf, 1.0 - ADAPTIVE_FRACTION))
    k_node = np.full(n, ADAPTIVE_K[1], dtype=np.int32)
    k_node[conf < low] = ADAPTIVE_K[0]
    k_node[conf > high] = ADAPTIVE_K[2]
    selected = np.where(np.arange(ADAPTIVE_K[2])[None, :] < k_node[:, None],
                        rank[:, :ADAPTIVE_K[2]], -1).astype(np.int32)
    adaptive = np.stack([T.mix_candidates(selected, raw_pop[m], k_first=ADAPTIVE_K[2],
                                         k=K_LIST, fill=rank)
                         for m in range(W)]).astype(np.int32)
    adaptive_list_s = time.perf_counter() - start

    start = time.perf_counter()
    xyf = np.asarray(xy, dtype=np.float64)
    span = float((xyf.max(0) - xyf.min(0)).max()) or 1.0
    xyn = (xyf - xyf.min(0)) / span
    count_nb, count_d = M.knn_np(xyn, K_GRAPH)
    extra = int(MODEL.cfg.get("graph_extra", 0))
    if extra:
        count_nb, count_d, _ = M.extend_graph(xyn, count_nb, succ, extra)
    assert np.array_equal(count_nb, prediction["NB"]), "Sıklık ve model grafikleri farklı"
    freq = T.ensemble_frequency(count_nb, succ)
    freq_rank = T.frequency_candidates(count_nb, count_d, freq, k=K_GRAPH)
    frequency = np.stack([T.mix_candidates(freq_rank, raw_pop[m], k_first=K_LEARNED,
                                           k=K_LIST, fill=freq_rank[:, K_LEARNED:])
                          for m in range(W)]).astype(np.int32)
    frequency_s = time.perf_counter() - start

    for arr in [pop5, frequency, betatour, adaptive]:
        valid_candidates(arr, n)
    meta = dict(successor_build_s=float(successor_s), pop5_build_s=float(pop5_s),
                betatour_build_s=float(prediction_s + betatour_list_s),
                betatour_adaptive_build_s=float(prediction_s + adaptive_list_s),
                adaptive_list_s=float(adaptive_list_s), frequency_build_s=float(frequency_s),
                restart_pop_raw_overhead_s=0.0,
                restart_pop_5_overhead_s=float(pop5_s),
                betatour_overhead_s=float(successor_s + prediction_s + betatour_list_s),
                betatour_adaptive_overhead_s=float(successor_s + prediction_s + adaptive_list_s),
                frequency_overhead_s=float(successor_s + frequency_s),
                model_predict_s=float(prediction["t_total"]),
                adaptive_k2_nodes=int((k_node == 2).sum()),
                adaptive_k3_nodes=int((k_node == 3).sum()),
                adaptive_k4_nodes=int((k_node == 4).sum()),
                adaptive_changed_node_frac=float(np.any(adaptive != betatour, axis=2).mean()),
                raw_pop_candidates_per_node=float((raw_pop >= 0).sum(axis=2).mean()),
                betatour_top3_in_original_pop5=float(np.mean([
                    (betatour[m, :, :K_LEARNED, None] == raw_pop[m, :, None, :]).any(-1).mean()
                    for m in range(W)])))
    return dict(restart_pop_raw=raw_pop.astype(np.int32), restart_pop_5=pop5,
                frequency=frequency, betatour=betatour, betatour_adaptive=adaptive), meta

def make_resumed_arm(inst, phase, phase_meta, candidate_lists, overhead_s, arm_name):
    budget = inst["budget"]
    prefix = float(phase_meta["wall_s"]) + float(overhead_s)
    remaining = budget - prefix
    if remaining <= 0.25:
        runs = []
        for m, seed in enumerate(SEEDS):
            first = phase_meta["first_events"][m]
            best = best_within(first, budget)
            assert best is not None, f"{arm_name} / {seed}: ilk tur bütçe dışında"
            runs.append(dict(seed=seed, events=first, first_events=first, continuation_events=[],
                             best_at_budget=int(best), wall_s=0.0, returncode=None, timed_out=False))
        return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                    continuation_wall_s=0.0, conceptual_prelude_s=float(prefix),
                    continuation_budget_s=max(0.0, float(remaining)), skipped_due_budget=True)
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_guided, [dict(common, budget=remaining, seed=SEEDS[m],
                                             cand=candidate_lists[m], init_tour=phase["tours"][m],
                                             timeout=remaining + PAD_SECONDS)
                                          for m in range(W)])
    runs = []
    for m, (seed, res) in enumerate(zip(SEEDS, raw)):
        assert (res["returncode"] in (0, None) and
                (res["returncode"] == 0 or res["timed_out"])), \
            f"{arm_name} / {seed}: LKH çıkış kodu {res['returncode']}"
        first = phase_meta["first_events"][m]
        resumed = event_list(res, prefix + res["t_start"] - start)
        if res["cost_check"] is not None and res["returncode"] == 0:
            resumed.append([int(res["cost_check"]),
                            float(prefix + res["t_start"] - start + res["wall"])])
        events = sorted(first + resumed, key=lambda e: e[1])
        best = best_within(events, budget)
        assert best is not None, f"{arm_name} / {seed}: bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, first_events=first, continuation_events=resumed,
                         best_at_budget=int(best), wall_s=float(res["wall"]),
                         returncode=res["returncode"], timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                continuation_wall_s=float(time.time() - start),
                conceptual_prelude_s=float(prefix), continuation_budget_s=float(remaining))

def load_phase(stage_dir):
    meta_path, array_path = stage_dir / "phase1.json", stage_dir / "phase1.npz"
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def load_lists(stage_dir, name):
    meta_path, array_path = stage_dir / "candidates.json", OUTPUT_ROOT / "candidates" / (name + ".npz")
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def run_one_instance(inst, ordinal):
    name = inst["name"]
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    if final_path.exists():
        data = json.loads(final_path.read_text())
        assert data.get("complete") is True and set(data["arms"]) == set(ARM_NAMES)
        print("ATLA (tamam):", name, flush=True)
        return data
    stage_dir = OUTPUT_ROOT / "stages" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab VM: yarım kalan örnek aynı donanımda baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            (OUTPUT_ROOT / "candidates" / (name + ".npz")).unlink(missing_ok=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    xy_sha = hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()
    full_path = stage_dir / "pop_full.json"
    full = json.loads(full_path.read_text()) if full_path.exists() else None
    controls = ["restart_pop_raw", "restart_pop_5", "frequency", "betatour", "betatour_adaptive"]
    rng = np.random.default_rng(10000 + ordinal)
    control_order = [str(x) for x in rng.permutation(controls)]

    def ensure_full():
        nonlocal full
        if full is None:
            print(name, "/ kesintisiz POPMUSIC", flush=True)
            full = make_full_arm(inst)
            write_json_atomic(full_path, full)

    def ensure_controls():
        phase_loaded = load_phase(stage_dir)
        if phase_loaded is None:
            print(name, "/ altı ortak ilk deneme", flush=True)
            phase, phase_meta = make_phase1(inst)
            write_npz_atomic(stage_dir / "phase1.npz", **phase)
            write_json_atomic(stage_dir / "phase1.json", phase_meta)
        else:
            phase, phase_meta = phase_loaded
        lists_loaded = load_lists(stage_dir, name)
        if lists_loaded is None:
            print(name, "/ aday listeleri", flush=True)
            lists, list_meta = make_candidate_lists(inst, phase)
            write_npz_atomic(OUTPUT_ROOT / "candidates" / (name + ".npz"), **lists)
            write_json_atomic(stage_dir / "candidates.json", list_meta)
        else:
            lists, list_meta = lists_loaded
        for arm in control_order:
            path = stage_dir / (arm + ".json")
            if path.exists():
                continue
            print(name, "/", arm, flush=True)
            result = make_resumed_arm(inst, phase, phase_meta, lists[arm],
                                      list_meta[arm + "_overhead_s"], arm)
            write_json_atomic(path, result)
        return phase_meta, list_meta

    if ordinal % 2 == 0:
        ensure_full()
        phase_meta, list_meta = ensure_controls()
    else:
        phase_meta, list_meta = ensure_controls()
        ensure_full()
    arms = {arm: json.loads((stage_dir / (arm + ".json")).read_text()) for arm in ARM_NAMES}
    first_match = [int(arms["pop_full"]["runs"][m]["events"][0][0]) ==
                   int(phase_meta["first_events"][m][0][0]) for m in range(W)]
    data = dict(schema=1, complete=True, name=name, group=inst["group"], n=len(inst["xy"]),
                budget_s=float(inst["budget"]), optimum=inst["optimum"], ew=inst["ew"],
                xy_sha256=xy_sha, seeds=SEEDS, control_order=control_order,
                full_first=bool(ordinal % 2 == 0), hardware=HARDWARE,
                phase1=phase_meta, overheads=list_meta,
                first_trial_matches_uninterrupted=first_match, arms=arms)
    write_json_atomic(final_path, data)
    b = {arm: arms[arm]["best_at_budget"] for arm in ARM_NAMES}
    print(f"TAMAM {name}: full={b['pop_full']}, raw_restart={b['restart_pop_raw']}, "
          f"restart5={b['restart_pop_5']}, frequency={b['frequency']}, "
          f"BetaTour={b['betatour']}, adaptive={b['betatour_adaptive']}", flush=True)
    return data


## 5. Deneyi çalıştır / kaldığı yerden sürdür

Tahmini 7–10 saatlik çalışmayı birkaç Colab oturumuna bölebilirsiniz. Drive'a her aşama atomik yazılır. Kesilirse aynı GPU/vCPU türüyle en baştan Tümünü çalıştır'ı seçin. Tamamlanan örnekler atlanır; MAX_NEW_INSTANCES = 8 gibi bir sınır oturum başına yükü azaltır.


In [ ]:
def warmup():
    xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
    succ = np.stack([np.roll(np.arange(100), -1)] * W, axis=1)
    M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=np.ones(W))
    first = T.task_first_trial(LKH, str(WORK_ROOT / "lkh_tmp"), SEEDS[0], xy,
                               timeout=30)
    assert T.is_valid_tour(first["tour"], 100) and first["cand"].shape == (100, K_LIST)
    print("GPU ve LKH öz sınaması tamam.")

warmup()

def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES),
                           finished=finished, last_instance=last_instance,
                           updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))

save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == hashlib.sha256(
            np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest(), \
            f"{inst['name']}: eski kayıt mevcut kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
save_progress(state="complete" if done == len(INSTANCES) else "partial")
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time()-session_start)/3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")


GPU ve LKH öz sınaması tamam.
ATLA (tamam): adaptive_clu1k_000
ATLA (tamam): adaptive_clu1k_001
ATLA (tamam): adaptive_clu1k_002
ATLA (tamam): adaptive_clu1k_003
ATLA (tamam): adaptive_clu1k_004
ATLA (tamam): adaptive_clu1k_005
ATLA (tamam): adaptive_clu1k_006
ATLA (tamam): adaptive_clu1k_007
ATLA (tamam): adaptive_uni1k_000
ATLA (tamam): adaptive_uni1k_001
ATLA (tamam): adaptive_uni1k_002
ATLA (tamam): adaptive_uni1k_003
ATLA (tamam): adaptive_uni1k_004
ATLA (tamam): adaptive_uni1k_005
ATLA (tamam): adaptive_uni1k_006
ATLA (tamam): adaptive_uni1k_007
ATLA (tamam): adaptive_clu10k_000
ATLA (tamam): adaptive_clu10k_001
ATLA (tamam): adaptive_clu10k_002
Yeni Colab VM: yarım kalan örnek aynı donanımda baştan çalıştırılıyor: adaptive_clu10k_003
adaptive_clu10k_003 / altı ortak ilk deneme
adaptive_clu10k_003 / aday listeleri
adaptive_clu10k_003 / frequency
adaptive_clu10k_003 / restart_pop_raw
adaptive_clu10k_003 / restart_pop_5
adaptive_clu10k_003 / betatour_adaptive
adaptive_clu10k_003 / 

## 6. Önceden belirlenen örnek düzeyi analiz

Birincil sonuç tam 40 yeni örnekte uyarlamalı / sabit BetaTour en iyi-altı tur maliyetidir. Negatif yüzde fark uyarlamalı liste lehinedir. İki yönlü kesin işaret testi (p < 0,05) eşitlikleri çıkarır; 5.000 örnek düzeyi bootstrap tekrarından %95 ortalama fark aralığı hesaplanır. Diğer yedi genel karşılaştırma keşifseldir; Holm sütunu sekiz karşılaştırmayı birlikte düzeltir. Alt gruplar betimleyicidir. Eksik örnek varsa rapor ara sonuçtur ve p değeri hesaplanmaz.

Kesintisiz ve yeniden başlatılan kolların ilk tur uzunlukları bir örnekte bile eşleşmezse ham sonuçlar saklanır, fakat çıkarımsal p değeri üretilmez.


In [ ]:
from scipy.stats import binomtest
from IPython.display import display

def load_finished():
    by_name = {x["name"]: x for x in INSTANCES}
    records = []
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        if item.get("complete") and item["name"] in by_name:
            assert set(item["arms"]) == set(ARM_NAMES)
            assert all(len(item["arms"][arm]["runs"]) == W for arm in ARM_NAMES)
            records.append(item)
    return records

records = load_finished()
mismatch = [r["name"] for r in records if not all(r["first_trial_matches_uninterrupted"])]
rows, seed_rows = [], []
for item in records:
    b = float(item["budget_s"])
    row = dict(instance=item["name"], group=item["group"], n=item["n"], budget_s=b,
               optimum=item["optimum"], first_match_all=all(item["first_trial_matches_uninterrupted"]),
               phase1_s=item["phase1"]["wall_s"], model_s=item["overheads"]["model_predict_s"],
               top3_in_pop5=item["overheads"]["betatour_top3_in_original_pop5"],
               raw_pop_candidates_per_node=item["overheads"]["raw_pop_candidates_per_node"],
               adaptive_changed_node_frac=item["overheads"]["adaptive_changed_node_frac"],
               cpu_model=item["hardware"]["cpu_model"], gpu=item["hardware"]["gpu"])
    for arm in ARM_NAMES:
        arm_data = item["arms"][arm]
        row[arm] = int(arm_data["best_at_budget"])
        for run in arm_data["runs"]:
            seed_rows.append(dict(instance=item["name"], group=item["group"], n=item["n"],
                                  seed=run["seed"], arm=arm, cost=run["best_at_budget"],
                                  budget_s=b))
    for treatment, control, label in [
        ("betatour_adaptive", "betatour", "adaptive_vs_fixed_pct"),
        ("betatour_adaptive", "restart_pop_5", "adaptive_vs_pop5_pct"),
        ("betatour_adaptive", "frequency", "adaptive_vs_frequency_pct"),
        ("betatour_adaptive", "pop_full", "adaptive_vs_full_pct"),
        ("betatour", "restart_pop_5", "fixed_vs_pop5_pct"),
        ("betatour", "frequency", "fixed_vs_frequency_pct"),
        ("betatour", "pop_full", "fixed_vs_full_pct"),
        ("restart_pop_raw", "pop_full", "raw_restart_vs_full_pct"),
    ]:
        row[label] = 100.0 * (row[treatment] - row[control]) / row[control]
    for arm in ARM_NAMES:
        if item["optimum"] is not None:
            row[arm + "_opt_gap_pct"] = 100.0 * (row[arm] - item["optimum"]) / item["optimum"]
    rows.append(row)

per_instance = pd.DataFrame(rows)
per_seed = pd.DataFrame(seed_rows)
per_instance.to_csv(OUTPUT_ROOT / "summary" / "per_instance.csv", index=False)
per_seed.to_csv(OUTPUT_ROOT / "summary" / "per_seed.csv", index=False)

COMPARISONS = {
    "Uyarlamalı / sabit BetaTour [birincil]": "adaptive_vs_fixed_pct",
    "Uyarlamalı / eşit-5 POP": "adaptive_vs_pop5_pct",
    "Uyarlamalı / tur sıklığı": "adaptive_vs_frequency_pct",
    "Uyarlamalı / kesintisiz POP": "adaptive_vs_full_pct",
    "Sabit BetaTour / eşit-5 POP": "fixed_vs_pop5_pct",
    "Sabit BetaTour / tur sıklığı": "fixed_vs_frequency_pct",
    "Sabit BetaTour / kesintisiz POP": "fixed_vs_full_pct",
    "Ham POP yeniden başlatma / kesintisiz POP": "raw_restart_vs_full_pct",
}

def paired_stats(values, bootstrap_seed, inferential):
    x = np.asarray(values, dtype=float)
    assert np.isfinite(x).all() and len(x)
    wins, ties, losses = int((x < -1e-12).sum()), int((abs(x) <= 1e-12).sum()), int((x > 1e-12).sum())
    if inferential:
        p = float(binomtest(wins, wins + losses, 0.5).pvalue) if wins + losses else 1.0
        rng = np.random.default_rng(bootstrap_seed)
        means = x[rng.integers(0, len(x), size=(5000, len(x)))].mean(axis=1)
        lo, hi = (float(v) for v in np.quantile(means, [0.025, 0.975]))
    else:
        p, lo, hi = np.nan, np.nan, np.nan
    return dict(instances=len(x), win=wins, tie=ties, loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=lo, ci95_high_pct=hi, sign_p=p)

inference_ready = len(INSTANCES) > 0 and len(records) == len(INSTANCES) and not mismatch
stats_rows = []
if inference_ready:
    for group, frame in [("all", per_instance)] + list(per_instance.groupby("group", sort=True)):
        for i, (label, column) in enumerate(COMPARISONS.items()):
            stats_rows.append(dict(group=group, comparison=label,
                                   **paired_stats(frame[column].to_numpy(), 20261025 + i, group == "all")))
stats = pd.DataFrame(stats_rows)
if len(stats):
    stats["sign_p_holm"] = np.nan
    for group, positions in stats.groupby("group").groups.items():
        if group != "all":
            continue
        ix = list(positions)
        ordered = sorted(ix, key=lambda j: stats.loc[j, "sign_p"])
        running = 0.0
        for rank, j in enumerate(ordered):
            running = max(running, (len(ordered) - rank) * float(stats.loc[j, "sign_p"]))
            stats.loc[j, "sign_p_holm"] = min(1.0, running)
stats.to_csv(OUTPUT_ROOT / "summary" / "paired_stats.csv", index=False)

suite_complete = inference_ready
if mismatch:
    note = (f"GEÇERSİZ KARŞILAŞTIRMA: {len(mismatch)} örnekte ilk tur uzunlukları uyuşmuyor. "
            "Başlangıç eşleşmediği için çıkarımsal istatistikler üretilmedi; ham kayıtları inceleyin.")
elif not suite_complete:
    note = f"ARA SONUÇ: {len(records)}/{len(INSTANCES)} örnek tamamlandı. Tam 40 örneklik analiz için devam edin."
else:
    note = "40/40 yeni sentetik örnek tamamlandı. Birincil karşılaştırma uyarlamalı / sabit BetaTour'dur."
cpu_models = sorted({r["hardware"]["cpu_model"] for r in records})
lines = ["# BetaTour güven uyarlamalı aday listesi deneyi", "", note, "",
         "Yüzde fark = 100 × (ilk kol − ikinci kol) / ikinci kol; negatif fark ilk kol lehine.",
         "Birincil karşılaştırma uyarlamalı / sabit BetaTour'dur; tam 40 örnek için iki yönlü kesin işaret testi ve p < 0.05 eşiği önceden belirlendi.",
         "Güven kuralı: alt çeyrekte 2, orta yarıda 3, üst çeyrekte 4 model adayı; POPMUSIC ile beşe tamamla.",
         "Aynı ağ ağırlıkları, ilk turlar, altı LKH tohumu, beş aday ve dış aday dosyası biçimi kullanılır.",
         "Diğer karşılaştırmalar ve alt gruplar keşifseldir; Holm sütunu sekiz genel karşılaştırmayı düzeltir.",
         "Güven aralığı örnek düzeyinde 5000 bootstrap tekrarından, p değeri iki yönlü işaret testinden gelir.",
         "Eksik örneklerde veya ilk turlar eşleşmezse çıkarımsal p değeri üretilmez.",
         "Ortak ilk aşama ve her kolun hazırlık/devam süresi çizelgeye katılır; Drive kayıt süresi dışarıdadır.",
         "Ham POP / kesintisiz POP farkı aday dosyası ve cezalar da değiştiği için saf yeniden başlatma etkisi değildir.", ""]
if len(cpu_models) > 1:
    lines += ["UYARI: örnekler birden fazla CPU modelinde çalışmış: " + ", ".join(cpu_models), ""]
if mismatch:
    lines += [f"UYARI: {len(mismatch)} örnekte kesintisiz ve durdurulmuş koşunun ilk maliyeti uyuşmadı: "
              + ", ".join(mismatch), ""]
if len(stats):
    lines += ["```", stats.round(5).to_string(index=False), "```", ""]
lines += ["Ham örnek ve tohum tabloları: summary/per_instance.csv, summary/per_seed.csv.",
          "Aday listeleri ve büyük aşama kayıtları Drive'daki candidates/ ve stages/ klasörlerindedir; varsayılan ZIP bunların .npz dosyalarını içermez."]
(OUTPUT_ROOT / "summary" / "RAPOR.md").write_text("\n".join(lines), encoding="utf-8")
print(note)
if mismatch:
    print("İlk deneme uyuşmazlıkları:", mismatch)
if len(stats):
    display(stats[stats["group"] == "all"].round(5))


40/40 yeni sentetik örnek tamamlandı. Birincil karşılaştırma uyarlamalı / sabit BetaTour'dur.


,group,comparison,instances,win,tie,loss,mean_delta_pct,median_delta_pct,ci95_low_pct,ci95_high_pct,sign_p,sign_p_holm
0,all,Uyarlamalı / sabit BetaTour [birincil],40,12,14,14,-0.00008,0.00000,-0.00262,0.00243,0.84502,1.00000
1,all,Uyarlamalı / eşit-5 POP,40,20,14,6,-0.00539,-0.00028,-0.00845,-0.00260,0.00936,0.07484
2,all,Uyarlamalı / tur sıklığı,40,13,14,13,-0.00100,0.00000,-0.00338,0.00114,1.00000,1.00000
3,all,Uyarlamalı / kesintisiz POP,40,12,15,13,0.00240,0.00000,-0.00054,0.00565,1.00000,1.00000
4,all,Sabit BetaTour / eşit-5 POP,40,17,15,8,-0.00531,0.00000,-0.00880,-0.00182,0.10775,0.64651
5,all,Sabit BetaTour / tur sıklığı,40,15,15,10,-0.00092,0.00000,-0.00367,0.00200,0.42436,1.00000
6,all,Sabit BetaTour / kesintisiz POP,40,11,14,15,0.00248,0.00000,-0.00072,0.00588,0.55720,1.00000
7,all,Ham POP yeniden başlatma / kesintisiz POP,40,7,13,20,0.00703,0.00093,0.00367,0.01068,0.01916,0.13410


## 7. İsteğe bağlı aday kapsaması

Yeni 40 örneğin önceden hazırlanmış bağımsız referans turları yoktur. Örnek adıyla NPY tur dosyalarını MyDrive/BetaTour/adaptive40_v1/references içine koyarsanız aday kapsaması hesaplanır. İsteğe bağlı 20k referans üretimi uzun sürer ve birincil sonlanımı değiştirmez.


In [ ]:
# Optional coverage for this new holdout; only local reference tours.
GENERATE_20K_REFERENCES = False  # change to True, rerun this cell, then rerun the ZIP cell

def load_reference_tours():
    refs = {}
    for path in sorted((OUTPUT_ROOT / "references").glob("*.npy")):
        tour = np.load(path, allow_pickle=False)
        refs[path.stem] = (tour, "independent_lkh")
    return refs

references = load_reference_tours()
if GENERATE_20K_REFERENCES:
    finished_names = {item["name"] for item in records}
    for inst in INSTANCES:
        if (inst["group"] not in {"uni20k", "clu20k"} or
                inst["name"] not in finished_names or inst["name"] in references):
            continue
        print("Bağımsız 20k referans turu:", inst["name"], flush=True)
        ref_budget = 5.0 * inst["budget"]
        result = T.lkh_solve(LKH, xy=inst["xy"], problem_file=inst["problem_file"], ew=inst["ew"],
                             params={**POP_PARAMS, "TIME_LIMIT": ref_budget,
                                     "MAX_TRIALS": 100_000_000, "SEED": 1001},
                             workroot=str(WORK_ROOT / "lkh_tmp"), timeout=ref_budget + PAD_SECONDS)
        assert T.is_valid_tour(result["tour"], len(inst["xy"])), f"Referans turu geçersiz: {inst['name']}"
        path = OUTPUT_ROOT / "references" / (inst["name"] + ".npy")
        tmp = path.with_name(path.stem + ".tmp.npy")
        np.save(tmp, result["tour"].astype(np.int32))
        os.replace(tmp, path)
        references[inst["name"]] = (result["tour"], "independent_lkh")

coverage_rows = []
missing_reference = []
for item in records:
    name, n = item["name"], int(item["n"])
    if name not in references:
        missing_reference.append(name)
        continue
    tour, source = references[name]
    assert T.is_valid_tour(tour, n), f"Referans turu boyutu yanlış: {name}"
    with np.load(OUTPUT_ROOT / "candidates" / (name + ".npz"), allow_pickle=False) as z:
        for arm in ["restart_pop_raw", "restart_pop_5", "frequency", "betatour", "betatour_adaptive"]:
            cand = z[arm]
            for m, seed in enumerate(SEEDS):
                missing = T.candidate_coverage(cand[m], tour, ks=[K_LIST])[K_LIST]["missing"]
                coverage_rows.append(dict(instance=name, group=item["group"], n=n, seed=seed,
                                          arm=arm, reference=source,
                                          missing_edges=int(missing), miss5_per_1000=1000.0 * missing / n))

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUTPUT_ROOT / "summary" / "candidate_coverage.csv", index=False)
print(f"Aday kapsaması: {len(coverage_rows)} tohum/kol satırı, "
      f"{len(missing_reference)} örnekte bağımsız referans turu eksik.")
if missing_reference:
    print("Eksik referanslar:", ", ".join(missing_reference))
if GENERATE_20K_REFERENCES:
    print("Yeni 20k referansları bağımsız POPMUSIC-5 çalışmalarıdır; başka yöntemlerin "
          "referanslarıyla aynı yöntem olarak yorumlamayın.")


Aday kapsaması: 0 tohum/kol satırı, 40 örnekte bağımsız referans turu eksik.
Eksik referanslar: adaptive_clu10k_000, adaptive_clu10k_001, adaptive_clu10k_002, adaptive_clu10k_003, adaptive_clu10k_004, adaptive_clu10k_005, adaptive_clu10k_006, adaptive_clu10k_007, adaptive_clu1k_000, adaptive_clu1k_001, adaptive_clu1k_002, adaptive_clu1k_003, adaptive_clu1k_004, adaptive_clu1k_005, adaptive_clu1k_006, adaptive_clu1k_007, adaptive_clu20k_000, adaptive_clu20k_001, adaptive_clu20k_002, adaptive_clu20k_003, adaptive_uni10k_000, adaptive_uni10k_001, adaptive_uni10k_002, adaptive_uni10k_003, adaptive_uni10k_004, adaptive_uni10k_005, adaptive_uni10k_006, adaptive_uni10k_007, adaptive_uni1k_000, adaptive_uni1k_001, adaptive_uni1k_002, adaptive_uni1k_003, adaptive_uni1k_004, adaptive_uni1k_005, adaptive_uni1k_006, adaptive_uni1k_007, adaptive_uni20k_000, adaptive_uni20k_001, adaptive_uni20k_002, adaptive_uni20k_003


## 8. ZIP ve otomatik indirme

ZIP; manifesti, ham örnek JSON sonuçlarını, aşama meta verilerini, analiz CSV'lerini ve raporu içerir. Aday matrisleri ve büyük aşama dizileri Drive'da kalır; varsayılan ZIP bunları içermez ve tek başına devam noktası değildir. Tam arşiv isterseniz INCLUDE_CANDIDATES_IN_ZIP = True yapın. Son hücre tamamlandığında ZIP Drive'a kaydedilir ve tarayıcı indirmesi başlar.


In [ ]:
import shutil
import zipfile

zip_name = EXPERIMENT_NAME + "_results.zip"
local_zip = WORK_ROOT / zip_name
with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    included = 0
    for path in OUTPUT_ROOT.rglob("*"):
        if not path.is_file() or path.suffix == ".tmp" or path.name.endswith(".tmp.npz"):
            continue
        if path.suffix == ".zip":
            continue
        relative = path.relative_to(OUTPUT_ROOT)
        if not INCLUDE_CANDIDATES_IN_ZIP and relative.parts[0] in {"candidates", "references"}:
            continue
        if not INCLUDE_CANDIDATES_IN_ZIP and path.suffix == ".npz":
            continue
        z.write(path, str(relative))
        included += 1

drive_zip = OUTPUT_ROOT / zip_name
shutil.copy2(local_zip, drive_zip)
print(f"ZIP hazır: {drive_zip} ({local_zip.stat().st_size / 1024**2:.1f} MB, {included} dosya)")
if SAVE_TO_DRIVE:
    from google.colab import files
    files.download(str(local_zip))  # browser download starts automatically
else:
    print("Drive kapalı; yerel ZIP:", local_zip)
